In [ ]:
import sys; sys.path.append('..')
import numpy as np, importlib
import mesh, fd_validation, parallelism

In [ ]:
np.random.seed(0)
m = mesh.Mesh('../../misc/examples/meshes/square_hole.off',  embeddingDimension=3)
m = mesh.Mesh('../../misc/examples/meshes/sphere_hires.msh', embeddingDimension=3)

# Elastic sheet loads

In [ ]:
import elastic_sheet, energy, tensors, loads
psi = energy.NeoHookeanYoungPoisson(2, 0, 0.3)
es = elastic_sheet.ElasticSheet(m, psi)

In [ ]:
gravity = loads.Gravity(es, 1.0)
spreaders = loads.Spreaders(es, [[0, 1], [2, 3]], [[0, 1]], 100)
springs = loads.Springs(es, [loads.AttachmentPointCoordinate([0], [1.0]), loads.AttachmentPointCoordinate([3, 6, 9], [1/3, 1/3, 1/3])], # Pull the x component of vertex 0, and the x component of the barycenter of vertices 2, 3, 4...
                            [loads.AttachmentPointCoordinate([3], [1.0]), loads.AttachmentPointCoordinate(0.0)],  # ... toward the x component of vertex 1, and the fixed value 0, respectively
                        1.0)

In [ ]:
prob = es.EquilibriumProblem([spreaders, gravity, springs])

In [ ]:
es.setVars(es.getVars() + 1e-2 * np.random.uniform(size=es.numVars()))
es.updateParametrization()

In [ ]:
fd_validation.gradConvergencePlot(prob)

In [ ]:
fd_validation.hessConvergencePlot(prob)

# Elastic solid loads

In [ ]:
import elastic_solid
m = mesh.Mesh('../../misc/examples/meshes/ball.msh')

In [ ]:
import tri_mesh_viewer
v = tri_mesh_viewer.TetMeshViewer(m, wireframe=True)
v.tetShrinkFactor = 0.5
v.show()

In [ ]:
psi = energy.NeoHookeanYoungPoisson(3, 0, 0.3)
es = elastic_solid.ElasticSolid(m, psi)
es.setVars(es.getVars() + 1e-2 * np.random.uniform(size=es.numVars()))

In [ ]:
gravity = loads.Gravity(es, 1.0)
sfit = loads.SphereFitter(es, 1.0, 1.0)
prob = es.EquilibriumProblem([sfit])

In [ ]:
v.tetShrinkFactor=0.1
G = sfit.grad_x().reshape((m.numNodes(), 3))
v.update(vectorField=-G)

In [ ]:
fd_validation.gradConvergencePlot(prob)

In [ ]:
np.random.seed(0)
fd_validation.hessConvergencePlot(prob)